In [1]:
!pip install -U transformers accelerate peft bitsandbytes -q

In [2]:
TARGET_MODEL = "mistralai/Mistral-7B-v0.1"  # Base model for sequence classification
DEBUG = False

In [3]:
from peft import get_peft_config, PeftModel, PeftConfig, get_peft_model, LoraConfig, TaskType
from transformers import BitsAndBytesConfig, AutoTokenizer, LlamaForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import torch
import numpy as np
import pandas as pd
import random
import os

In [4]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    
set_seed(42)

In [5]:
# Load competition data
train = pd.read_csv('/kaggle/input/name-that-llm/train.csv') if os.path.exists('/kaggle/input/name-that-llm/train.csv') else pd.read_csv('input/train.csv')

# Define the 18 valid model labels from the competition rules
VALID_MODELS = [
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5",
    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3"
]

model_to_id = {model: i for i, model in enumerate(VALID_MODELS)}
id_to_model = {i: model for i, model in enumerate(VALID_MODELS)}

# Map string labels to integers if your target column contains model names
if train['model'].dtype == 'object':
    train['target'] = train['model'].map(model_to_id)

In [6]:
# GroupKFold setup based on prompts/questions if available, or standard StratifiedKFold
from sklearn.model_selection import StratifiedKFold
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train['fold'] = -1
for i, (train_index, test_index) in enumerate(folds.split(train, train['target'])):
    train.loc[test_index, 'fold'] = i

In [7]:
# Format input text passage for classification
train['input'] = "Passage: " + train['text']
train = train.rename(columns={'target': 'label'})

In [8]:
tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples, max_length=512):
    return tokenizer(examples["input"], truncation=True, max_length=max_length, padding=True)

from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

In [9]:
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e_x / e_x.sum(axis=1, keepdims=True)

!mkdir -p llm_models

In [10]:
# Training Loop across 5 Folds
NUM_LABELS = len(VALID_MODELS)

for fold in range(5):
    print(f'--- Training Fold {fold+1}/5 ---')
    valid_df = train[train["fold"] == fold]
    train_df = train[train["fold"] != fold]
    
    train_ds = Dataset.from_pandas(train_df)
    valid_ds = Dataset.from_pandas(valid_df)
    
    train_tokenized_ds = train_ds.map(preprocess_function, batched=True)
    valid_tokenized_ds = valid_ds.map(preprocess_function, batched=True)
    
    peft_config = LoraConfig(
        r=8,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    )

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    base_model = LlamaForSequenceClassification.from_pretrained(
        TARGET_MODEL,
        num_labels=NUM_LABELS,
        quantization_config=bnb_config,
        device_map={"": 0}
    )
    base_model.config.pad_token_id = tokenizer.pad_token_id
    model = get_peft_model(base_model, peft_config)
    
    training_args = TrainingArguments(
        output_dir=f"llm_models/fold{fold}",
        learning_rate=2e-4,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        optim='paged_adamw_32bit',
        lr_scheduler_type="cosine",
        num_train_epochs=2,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        report_to='none'
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized_ds,
        eval_dataset=valid_tokenized_ds,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    
    trainer.train()
    
    # Clean up memory between folds
    del trainer, model, base_model
    torch.cuda.empty_cache()

In [11]:
# Inference and Submission Generation matching 'Name That LLM' constraints
test_path = '/kaggle/input/competitions/name-that-llm/test.csv' if os.path.exists('/kaggle/input/competitions/name-that-llm/test.csv') else 'input/test.csv'
test = pd.read_csv(test_path)
test['input'] = "Passage: " + test['text']

test_ds = Dataset.from_pandas(test)
test_tokenized_ds = test_ds.map(preprocess_function, batched=True)

final_preds = []
for fold in range(5):
    print(f'Inference for Fold {fold}')
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    base_model = LlamaForSequenceClassification.from_pretrained(
        TARGET_MODEL,
        num_labels=NUM_LABELS,
        quantization_config=bnb_config,
        device_map={"": 0}
    );
    base_model.config.pad_token_id = tokenizer.pad_token_id
    
    model = PeftModel.from_pretrained(base_model, f"llm_models/fold{fold}/checkpoint-best") # update checkpoint folder dynamically if needed
    trainer = Trainer(model=model, tokenizer=tokenizer, data_collator=data_collator)
    
    pred_output = trainer.predict(test_tokenized_ds)
    probas = softmax(pred_output.predictions)
    final_preds.append(probas)
    
    del trainer, model, base_model
    torch.cuda.empty_cache()

# Average probabilities across all 5 folds
mean_probas = np.mean(final_preds, axis=0)
predicted_class_indices = np.argmax(mean_probas, axis=1)
predictions = [id_to_model[idx] for idx in predicted_class_indices]

# Create submission file as parquet per competition rules
submission = pd.DataFrame({'id': test['id'], 'model': predictions})
submission.to_parquet('submission.parquet', index=False)
print("Submission successfully saved to submission.parquet!")